# Классификация заемщиков линейными моделями

## курс "Машинное обучение 1", программа AIMasters, 2025

## Студент: Фоменко Елизавета Антоновна


## Реализация алгоритмов (5 баллов)

Ниже нужно написать собственную реализацию линейного классификатора с произвольной функцией потерь и реализацию функции и градиента функции потерь для логистической регрессии.

Реализации можно частично проверить через юнит тесты.

В этом блоке можно использовать только `numpy, scipy`.


В `BinaryLogisticLoss` вам нужно реализовать расчет лосса и его градиента для функции
$$L(w) = \frac{1}{N} \sum_{N} [\log(1 + \exp(-y_i\langle w, x_i\rangle))] + \lambda \lVert w \rVert^2_2, \quad y \in \{-1, 1\}$$

- `func(self, X, y, w)` — вычисление значения функции потерь на матрице признаков X, векторе ответов y с вектором весов w.
- `grad(self, X, y, w)` — вычисление значения градиента функции потерь на матрице признаков X, векторе ответов y с вектором весов w.

У обоих методов одинаковые аргументы:
- X - выборка объектов
- y - вектор ответов
- w - вектор коэффициентов модели

Вектор коэффициентов имеет вид: w = `[bias, weights]`, то есть нулевой элемент w - `bias`, остальное - веса, участвующие в скалярном произведении. **Важно:** `bias` не участвует в расчете слагаемого с $\lambda$.

Обратите внимание, что на матрица X на входе без столбца с 1 в начале. Пример изменения Х внутри кода функций:
```python
X_new = np.c_[np.ones(X.shape[0]), X]
```

In [1]:
import numpy as np
import scipy
from scipy.special import expit
from scipy.special import logsumexp

class BinaryLogisticLoss():
    """
    Loss function for binary logistic regression.
    It should support l2 regularization.
    """

    def __init__(self, l2_coef):
        """
        Parameters
        ----------
        l2_coef - l2 regularization coefficient
        """
        self.l2_coef = l2_coef
        self.eps = 1e-8

    def func(self, X, y, w):
        """
        Get loss function value for data X, target y and coefficient w; w = [bias, weights].

        Parameters
        ----------
        X : numpy.ndarray
        y : 1d numpy.ndarray
        w : 1d numpy.ndarray

        Returns
        -------
        : float
        """
        # add columns of ones for bias
        X = np.c_[np.ones(X.shape[0]), X]

        # compute loss
        # matrix multiplication
        comp = -y * (X @ w)

        # as log(1 + exp(x)) -> x, x->inf, we dont compute it for large x (when toll becomes ~1e-11)
        idx = np.where(comp <= 25)[0]
        comp[idx] = np.log1p(np.exp(comp[idx]))
        lossfunc = comp.mean() + self.l2_coef * (w[1::]**2).sum()
        return lossfunc

    def grad(self, X, y, w):
        """
        Get loss function gradient for data X, target y and coefficient w; w = [bias, weights].

        Parameters
        ----------
        X : numpy.ndarray
        y : 1d numpy.ndarray
        w : 1d numpy.ndarray

        Returns
        -------
        : 1d numpy.ndarray
        """
        # add columns of ones for bias
        X = np.c_[np.ones(X.shape[0]), X]

        # calculate dL/dw
        grad = (-X * (y / (1 + np.exp(y * (X @ w)))).reshape(-1,1)).mean(axis=0) + 2 * self.l2_coef * np.hstack([0, w[1:]])
        return grad

In [2]:
loss_function = BinaryLogisticLoss(l2_coef=1.0)
X = np.array([
    [1, 2],
    [3, 4],
    [-5, 6]
])
y = np.array([-1, 1, 1])
w = np.array([1, 2, 3])
assert np.isclose(loss_function.func(X, y, w), 16.00008, atol=1e-5)

loss_function = BinaryLogisticLoss(l2_coef=0.0)
X = np.array([
    [10 ** 5],
    [-10 ** 5],
    [10 ** 5]
])
y = np.array([1, -1, 1])
w = np.array([1, 100])
assert np.isclose(loss_function.func(X, y, w), 0, atol=1e-5)

loss_function = BinaryLogisticLoss(l2_coef=0.0)
X = np.array([
    [10 ** 2],
    [-10 ** 2],
    [10 ** 2]
])
y = np.array([-1, 1, -1])
w = np.array([1, 100])
assert np.isclose(loss_function.func(X, y, w), 10000.333334, atol=1e-5)

loss_function = BinaryLogisticLoss(l2_coef=1.0)
X = np.array([
    [1, 2],
    [3, 4],
    [-5, 6]
])
y = np.array([-1, 1, 1])
w = np.array([1, 2, 3])
right_gradient = np.array([0.33325, 4.3335 , 6.66634])
assert np.isclose(loss_function.grad(X, y, w), right_gradient, atol=1e-5).all()

В `LinearModel` нужно реализовать линейную модель, поддерживающей обучение через стохастический градиентные спуск.

`__init__` — инициализатор класса с параметрами:
- loss_function — функция потерь, заданная классом
- batch_size — размер подвыборки, по которой считается градиент
- step_alpha — параметр шага градиентного спуска
- tolerance — критерий останова метода — модуль разности значений функции потерь на соседних итерациях метода меньше tolerance, не весов.
- max_iter — максимальное число итераций (эпох)

`fit(self, X, y, w_0=None)` — обучение линейной модели

- X — выборка объектов
- y — вектор ответов
- w_0 — начальное приближение вектора коэффициентов, если None, то необходимо инициализировать внутри метода. w_0 имеет вид `[bias_0, weights_0]`.

`predict_proba(self, X)` — получение вероятностей для 2х классов
- X — выборка объектов

Вы можете поменять формат изменения шага градиентного спуска, по дефолту предполагается, что можно использовать просто `step_alpha`.

Про `sgd`: нет необходимости проводить честное семплирование для каждого батча в методе стохасического градиентного спуска. Вместо этого предлагается в начале одной эпохи сгенерировать случайную перестановку индексов объектов, а затем последовательно выбирать объекты для нового батча из элементов этой перестановки. Псевдокод:
```python
epoch_rand_indexes = np.random.permutation(X.shape[0])
inner_cycle_length = int(np.ceil(X.shape[0] / self.batch_size))

for i in range(inner_cycle_length):
    start_index = self.batch_size * i
    finish_index = self.batch_size * (i + 1)
    batch_indexes = epoch_rand_indexes[start_index:finish_index]
    # тут считаем градиент только по batch_indexes
```

Еще несколько советов:

В промежуточных вычислениях стоит избегать вычисления $exp(−y_i⟨x_i,w⟩)$, иначе может произойти переполнение.
Вместо этого следует напрямую вычислять необходимые величины с помощью специализированных для этого функций: `np.logaddexp, scipy.special.logsumexp и scipy.special.expit`. В ситуации, когда вычисления экспоненты обойти не удаётся, можно воспользоваться процедурой «клипинга» (функция `numpy.clip`).


In [3]:
import numpy as np
from scipy.special import expit

class LinearModel:
    def __init__(
        self,
        loss_function,
        batch_size=100,
        step_alpha=1,
        tolerance=1e-5,
        max_iter=1000,
        random_seed=0,
        **kwargs
    ):
        """
        Parameters
        ----------
        loss_function : BaseLoss inherited instance
            Loss function to use
        batch_size : int
        step_alpha : float
        tolerance : float
            Tolerace for stop criterio.
        max_iter : int
            Max amount of epoches in method.
        """
        self.loss_function = loss_function
        self.batch_size = batch_size
        self.step_alpha = step_alpha
        self.tolerance = tolerance
        self.max_iter = max_iter
        self.random_seed = random_seed
        self.w = None

        np.random.seed(random_seed)

    def fit(self, X, y, w_0=None):
        """
        Parameters
        ----------
        X : numpy.ndarray or scipy.sparse.csr_matrix
            2d matrix, training set.
        y : numpy.ndarray
            1d vector, target values.
        w_0 : numpy.ndarray
            1d vector in binary classification.
            Initial approximation for SGD method - [bias, weights]
        """

        if w_0 is None:
            self.w = np.random.normal(0, 0.05, X.shape[1]+1) # [bias, weights]
        else:
            self.w = w_0

        prev_loss = float('inf')
        for epoch in range(self.max_iter):

            epoch_loss = 0

            epoch_rand_indexes = np.random.permutation(X.shape[0])
            inner_cycle_length = int(np.ceil(X.shape[0] / self.batch_size))

            for i in range(inner_cycle_length):
                # get batches
                batch_indexes = epoch_rand_indexes[self.batch_size * i : self.batch_size * (i + 1)]

                X_batch, y_batch = X[batch_indexes, :], y[batch_indexes]

                # get loss
                curr_loss = self.loss_function.func(X_batch, y_batch, self.w)
                epoch_loss += curr_loss

                # get gradients
                self.w -= self.step_alpha * self.loss_function.grad(X_batch, y_batch, self.w)

            # if |dLoss| < tol: terminate
            dLoss = np.abs(prev_loss - epoch_loss)
            prev_loss = epoch_loss
            if dLoss < self.tolerance:
                return self.w

            print(f'Epoch number: {epoch}; epoch loss: {epoch_loss:6f}, |dLoss| = {dLoss:.6f}')

        return self.w





    def predict_proba(self, X):
        """
        Parameters
        ----------
        X : numpy.ndarray or scipy.sparse.csr_matrix
            2d matrix, test set.
        Returns
        -------
        : numpy.ndarray
            probs, shape=(X.shape[0], 2)
        """
        X = np.c_[np.ones(X.shape[0]), X]
        logits = 1 / (1 + np.exp(X @ self.w))
        neg_logits = 1 - logits

        return np.vstack([logits, neg_logits]).T

In [4]:
# обратите внимание, что тут достаточно простой тест
# ниже еще есть проверка для данных из data
X1 = np.random.randint(1, 4, (1000, 10))
X2 = np.random.randint(-4, 0, (1000, 10))
X = np.vstack((X1, X2))
y = np.array([-1] * 1000 + [1] * 1000)
loss_function = BinaryLogisticLoss(l2_coef=0.1)
linear_model = LinearModel(
    loss_function=loss_function,
    batch_size=100,
    step_alpha=1,
    tolerance=1e-4,
    max_iter=1000,
)
linear_model.fit(X, y)
prediction_probs = linear_model.predict_proba(X)
predictions = (prediction_probs > 0.5).astype('int')[:, 1] * 2 - 1
assert np.isclose(predictions, y).all()

Epoch number: 0; epoch loss: 7.485981, |dLoss| = inf
Epoch number: 1; epoch loss: 1.066371, |dLoss| = 6.419610
Epoch number: 2; epoch loss: 1.053285, |dLoss| = 0.013086
Epoch number: 3; epoch loss: 1.046746, |dLoss| = 0.006539
Epoch number: 4; epoch loss: 1.043201, |dLoss| = 0.003545
Epoch number: 5; epoch loss: 1.041433, |dLoss| = 0.001767
Epoch number: 6; epoch loss: 1.040500, |dLoss| = 0.000934
Epoch number: 7; epoch loss: 1.040197, |dLoss| = 0.000302
Epoch number: 8; epoch loss: 1.040052, |dLoss| = 0.000145
Epoch number: 9; epoch loss: 1.039809, |dLoss| = 0.000244
Epoch number: 10; epoch loss: 1.039972, |dLoss| = 0.000163
Epoch number: 11; epoch loss: 1.040099, |dLoss| = 0.000126
Epoch number: 12; epoch loss: 1.039971, |dLoss| = 0.000128
Epoch number: 13; epoch loss: 1.039868, |dLoss| = 0.000103
Epoch number: 14; epoch loss: 1.039976, |dLoss| = 0.000108


## Эксперименты (5 баллов)

Эксперименты будем проводить на [датасете](https://www.kaggle.com/competitions/home-credit-default-risk/overview) по классификации заемщиков на плохих (target = 1: клиент с "payment difficulties") и хороших (target = 0: все остальные). Для экспериментов будем использовать лишь основной файл `application_train.csv`, а также перекодируем таргет в метки -1, 1.

Описание колонок находится в файле `description.csv`.

Для начала мы за вас считаем данные и поделим на обучение и тест.

Код в чтение, разбиение и предобработке менять не нужно.

In [5]:
from google.colab import drive
drive.mount('/content/drive/')

%cd /content/drive/My Drive/AI MASTERS/ML1

Mounted at /content/drive/
/content/drive/My Drive/AI MASTERS/ML1


In [6]:
# не меняем код
import pandas as pd
pd.options.display.max_columns = 100
pd.options.display.max_rows = 150


data = pd.read_csv('application_train.csv')
data.columns = [
    '_'.join([word.lower() for word in col_name.split(' ') if word != '-']) for col_name in data.columns
]
data.target = data.target.map({0: -1, 1: 1})
data.head(3)

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_type_suite,name_income_type,name_education_type,name_family_status,name_housing_type,region_population_relative,days_birth,days_employed,days_registration,days_id_publish,own_car_age,flag_mobil,flag_emp_phone,flag_work_phone,flag_cont_mobile,flag_phone,flag_email,occupation_type,cnt_fam_members,region_rating_client,region_rating_client_w_city,weekday_appr_process_start,hour_appr_process_start,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,reg_city_not_live_city,reg_city_not_work_city,live_city_not_work_city,organization_type,ext_source_1,ext_source_2,ext_source_3,apartments_avg,basementarea_avg,years_beginexpluatation_avg,years_build_avg,commonarea_avg,elevators_avg,...,apartments_medi,basementarea_medi,years_beginexpluatation_medi,years_build_medi,commonarea_medi,elevators_medi,entrances_medi,floorsmax_medi,floorsmin_medi,landarea_medi,livingapartments_medi,livingarea_medi,nonlivingapartments_medi,nonlivingarea_medi,fondkapremont_mode,housetype_mode,totalarea_mode,wallsmaterial_mode,emergencystate_mode,obs_30_cnt_social_circle,def_30_cnt_social_circle,obs_60_cnt_social_circle,def_60_cnt_social_circle,days_last_phone_change,flag_document_2,flag_document_3,flag_document_4,flag_document_5,flag_document_6,flag_document_7,flag_document_8,flag_document_9,flag_document_10,flag_document_11,flag_document_12,flag_document_13,flag_document_14,flag_document_15,flag_document_16,flag_document_17,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,...,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,-1,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,...,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,-1,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# не меняем код
test_idx = data.sk_id_curr % 10 >= 7
data_dict = dict()
data_dict['tst'] = data.loc[test_idx].reset_index(drop=True)
data_dict['tr'] = data.loc[~test_idx].reset_index(drop=True)

for key, df in data_dict.items():
    print(key, 'shape:', df.shape)

tst shape: (92221, 122)
tr shape: (215290, 122)


In [8]:
# не меняем код
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import numpy as np

features = data.select_dtypes(np.number).drop(columns=['target', 'sk_id_curr']).columns

X_tr, X_tst = data_dict["tr"][features].to_numpy(), data_dict["tst"][features].to_numpy()
y_tr, y_tst = data_dict["tr"]["target"].to_numpy(), data_dict["tst"]["target"].to_numpy()


prep = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

prep.fit(X_tr)

X_tr = prep.transform(X_tr)
X_tst = prep.transform(X_tst)

Инициализируйте написанный выше лосс и классификатор, для `BinaryLogisticLoss` возьмите параметр `l2_coef=0.1`, параметры `LinearModel` нужно подобрать так, чтобы [roc_auc_score](https://scikit-learn.org/1.5/modules/generated/sklearn.metrics.roc_auc_score.html) был больше 0.72.

In [9]:
loss_function = BinaryLogisticLoss(l2_coef=0.1)
clf = LinearModel(loss_function,
        batch_size=256,
        step_alpha=3e-4, max_iter = 100)

In [10]:
from sklearn.metrics import roc_auc_score

clf.fit(X_tr, y_tr)
roc_auc_score(y_tst, clf.predict_proba(X_tst)[:, 1])

Epoch number: 0; epoch loss: 652.908587, |dLoss| = inf
Epoch number: 1; epoch loss: 596.946698, |dLoss| = 55.961890
Epoch number: 2; epoch loss: 553.184522, |dLoss| = 43.762176
Epoch number: 3; epoch loss: 517.020454, |dLoss| = 36.164067
Epoch number: 4; epoch loss: 486.494169, |dLoss| = 30.526285
Epoch number: 5; epoch loss: 460.386481, |dLoss| = 26.107689
Epoch number: 6; epoch loss: 437.815569, |dLoss| = 22.570912
Epoch number: 7; epoch loss: 418.141053, |dLoss| = 19.674515
Epoch number: 8; epoch loss: 400.867468, |dLoss| = 17.273585
Epoch number: 9; epoch loss: 385.619276, |dLoss| = 15.248193
Epoch number: 10; epoch loss: 372.091321, |dLoss| = 13.527954
Epoch number: 11; epoch loss: 360.039194, |dLoss| = 12.052128
Epoch number: 12; epoch loss: 349.263541, |dLoss| = 10.775652
Epoch number: 13; epoch loss: 339.595965, |dLoss| = 9.667576
Epoch number: 14; epoch loss: 330.895647, |dLoss| = 8.700319
Epoch number: 15; epoch loss: 323.043654, |dLoss| = 7.851993
Epoch number: 16; epoch los

np.float64(0.7209012822657074)

In [11]:
assert roc_auc_score(y_tst, clf.predict_proba(X_tst)[:, 1]) > 0.72

Ура! Ваша модель что-то может :)

Теперь нужно поисследовать реализацию [LogisticRegression](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LogisticRegression.html) в sklearn.

Сравните различные `solver` по времени обучения/качеству на тесте. Напишите выводы.

Выбейте на тесте больше `0.737` roc_auc_score.
Для поиска лучшей модели можно использовать:
- свои наблюдения и интуицию
- [GridSearchCV](https://scikit-learn.org/dev/modules/generated/sklearn.model_selection.GridSearchCV.html)
- optuna


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [13]:
import time


# one-shot на параметрах кастомной модели
for solver in ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']:
    clf = LogisticRegression(penalty = 'l2', tol = 1e-4, C = 1/(3e-4), max_iter = 100, solver = solver, random_state = 0)
    try:
        start = time.perf_counter()
        clf.fit(X_tr, y_tr)
        end = time.perf_counter()
        rocauc = roc_auc_score(y_tst, clf.predict_proba(X_tst)[:, 1])
        print(f'Solver = {solver}; fit time: {end-start:.3f}; rocauc = {rocauc:.3f}')
    except:
        print(f'problem {solver} with L2 regularization')



#assert roc_auc_score(y_tst, clf.predict_proba(X_tst)[:, 1]) > 0.737

Solver = lbfgs; fit time: 3.734; rocauc = 0.735
Solver = liblinear; fit time: 76.942; rocauc = 0.738
Solver = newton-cg; fit time: 15.880; rocauc = 0.738
Solver = newton-cholesky; fit time: 2.086; rocauc = 0.738


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Solver = sag; fit time: 37.759; rocauc = 0.734
Solver = saga; fit time: 44.846; rocauc = 0.733


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [14]:
# смотрите, one-shot по параметрам кастомной модели дал нужный результат (rocauc = 0.738 > 0.737) сразу для трех солверов: liblinear, newton-cg и newton-cholesky :)

# теперь попробуем реализовать подбор гиперпараметров

In [ ]:
%pip install optuna
%pip install optuna-dashboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 9.9 MB/s eta 0:00:00


In [16]:
# к сожалению, liblinear и sag считаются слишком долго для использования optuna, поэтому я их не буду использовать

In [23]:
import optuna
import optuna.visualization as vis
from sklearn.model_selection import cross_val_score


# сначала оптимизируем солверы и lr именно для L2 регуляризации

# солверы liblinear, sag и saga  были исключены для optuna, так как они слишком долго работают (3 trials-a обрабатывались час?!). чтобы сделать побольше попыток я остановилась на быстроработающих солверах
def func4opt(trial):
    solver = trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'newton-cholesky'])
    C = trial.suggest_float('C', 5e3е, 4e5)
    clf = LogisticRegression(penalty = 'l2', tol = 1e-5, C = C, max_iter = 1000, solver = solver, random_state = 0, n_jobs=-1)
    scores = cross_val_score(clf, X_tr, y_tr, scoring='roc_auc', cv=3, n_jobs=-1)
    return scores.mean()


study = optuna.create_study(direction="maximize", study_name = 'best L2-reg model')
study.optimize(func4opt, n_trials=50)

print("Best hyperparameters:", study.best_params)
print("Best value:", study.best_value)
"""
vis.plot_optimization_history(study)
vis.plot_param_importances(study)
vis.plot_slice(study)"""

[I 2025-10-22 13:45:07,810] A new study created in memory with name: best L2-reg model
[I 2025-10-22 13:45:36,854] Trial 0 finished with value: 0.7352532473679708 and parameters: {'solver': 'newton-cg', 'C': 373476.93011714}. Best is trial 0 with value: 0.7352532473679708.
[I 2025-10-22 13:46:05,115] Trial 1 finished with value: 0.7351746545756767 and parameters: {'solver': 'lbfgs', 'C': 351474.40241027257}. Best is trial 0 with value: 0.7352532473679708.
[I 2025-10-22 13:46:08,881] Trial 2 finished with value: 0.7352596032344593 and parameters: {'solver': 'newton-cholesky', 'C': 251614.40139515503}. Best is trial 2 with value: 0.7352596032344593.
[I 2025-10-22 13:46:12,671] Trial 3 finished with value: 0.7352596110962661 and parameters: {'solver': 'newton-cholesky', 'C': 228713.23675504274}. Best is trial 3 with value: 0.7352596110962661.
[I 2025-10-22 13:46:41,074] Trial 4 finished with value: 0.7352533661747188 and parameters: {'solver': 'newton-cg', 'C': 395828.9788076195}. Best is

Best hyperparameters: {'solver': 'lbfgs', 'C': 257959.6679096309}
Best value: 0.7352904349146643


'\nvis.plot_optimization_history(study)\nvis.plot_param_importances(study)\nvis.plot_slice(study)'

In [18]:
# assert roc_auc_score(y_tst, clf.predict_proba(X_tst)[:, 1]) > 0.737

In [19]:
study

In [20]:
vis.plot_optimization_history(study)

In [21]:
vis.plot_param_importances(study)

In [22]:
vis.plot_slice(study)

Нарисуйте график `feature - weight`, показывающий `top_k` (на ваш выбор) признаков по модулю веса и их значения весов. <br>
Признаки должны идти по убыванию модуля веса. <br>
Лучше использовать [barplot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html) или аналоги из других библиотек. <br>
Опишите наблюдения, используя описания признаков в `description.csv`.

In [ ]:
description = pd.read_csv('description.csv', sep=';')
description = description[-description['Row'].isin(['TARGET', 'SK_ID_CURR'])]
description.shape

In [ ]:
import matplotlib.pyplot as plt

def plot_weights_logregr(model, features, description, top_k=25):
    '''
    рисует значения весов линейной модели при признаках

    top_k: рисовать первые top_k весов по модулю
    '''
    num_features_to_plot = min(top_k, len(features))
    weights, bias = model.coef_[0], model.intercept_[0]
    sorted_idx = np.argsort(-np.abs(weights))[:num_features_to_plot]

    features_to_plot = features[sorted_idx][::-1]

    fig, ax = plt.subplots(figsize=(12, num_features_to_plot / 2 - 1))
    ax2 = ax.twinx()

    weights_to_plot = weights[sorted_idx][::-1]

    colors = np.where(weights_to_plot > 0, 'palevioletred', 'cornflowerblue')

    container = ax.barh(y=features_to_plot, width=weights_to_plot, color = colors)

    ax.bar_label(container, weights_to_plot.round(3), color='black', fontsize=14)

    description.Row = description.Row.str.lower()
    descriptions = []
    for feature in features_to_plot:
        descriptions.append(description[description['Row'] == feature].Description.unique()[0])

    # настройка ах'a
    ax.margins(0.2, 0.05)
    ax.set_title(f'bias: {bias:.3f}', fontsize=18)
    ax.tick_params(axis='both', labelsize=14)
    ax.set_xlabel('weight', fontsize=18)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(np.arange(num_features_to_plot))
    ax2.set_yticklabels(descriptions, fontsize=12)

    plt.show()


plot_weights_logregr(clf, features, description, top_k=20)

Я решила дописать описание фичей прямо на график, чтобы удобнее было интерпретировать их значение. график стал чуть менее симпатичным, зато для ответа на вопрос задания нужно сделать меньше усилий



Довольно очевидно, что самыми влияющими факторами будет предполагаемый размер кредита. Чем он больше, тем выше веротность дефолта
При этом когда мы рассматриваем потребительские кредиты, то чем выше цена на приобретаемые товары, тем ниже вероятность дефолта. возможно, здесь логика такая: чем выше стоимость вещи, тем более состоятелен наш клиент.

Здесь же мы видим переменную ampt_annuity - грубо говоря, процент выплат и схема выплата долга нашим клиентом. Чем больше ставку по кредиту сделаем, тем больше вероятность получить задолженность и выйти в дефолт.

Далее мы видим разного рода информацию, которая нам позволяет судить о состоятельности клиента (например, информация о месте жительства клиента (цены на жилье в районе, задолженности по выплатам и тд), дате последней покупки телефона (чем больше времени прошло, тем, возможно, меньше средств на покупку нового), а также, что немаловажно, стабильность пребывания на рабочем месте (здесь тоже видим увеличение вероятности дефолта при увеличении времени работы, возможно, это значит, что наш клиент завис на одной работе... я не знаю тут).

Некоторые переменные объяснить не получится, так как они скорее всего являются внутренними банковскими - flag_documents и тд

Выведите топ признаков с наибольшим/наименьшим абсолютным весом.<br>
Опишите наблюдения (ответьте на вопрос: правда ли, что если признак `X` больше/меньше, то вероятность дефолта клиента выше/ниже?).

In [ ]:
# вывод топ фичей

In [ ]:
k = 10

weights = clf.coef_[0]

top_idx = np.argsort(-np.abs(weights))[:k]
end_idx = np.argsort(-np.abs(weights))[-k:]

top_weights = weights[top_idx]
top_features = features[top_idx]

end_weights = weights[end_idx]
end_features = features[end_idx]

print(f'Top {k} important features')
for _ in range(k):
    print(f'    {top_features[_].upper()} has weight = {top_weights[_]:.4f}.    {description[description['Row'] == top_features[_]].Description.unique()[0]}')

print(f'\nTop {k} least important features')
for _ in range(k):
    print(f'    {end_features[_].upper()} has weight = {end_weights[_]:.4f}.    {description[description['Row'] == end_features[_]].Description.unique()[0]}')


описание самых важных признаков смотрите выше

описаие наименее значимых признаков:

REG_CITY_NOT_WORK_CITY - если клиент работает не в городе проживания, значит, он, вероятно, не способен оплачивать проживание в городе работы => выше вероятность дефолта

DAYS_BIRTH - возраст на мой взгляд неоднозначно влияет на вероятность дефолта, тут скорее завсимость близка к квадратичой (слишком молодые и слишком старые - плохо)
остальные фичи объясниь сложно тк они внутренние. но описанные выше две фичи действительно согласно логике не должны влиять на вероятность дефолта или, во всяком случае, не должны влиять линейно (как в случае возраста)

